# Architecture

What is the underlying architecture of Kafi Streams?


## Overview

* [A dependency diagram](#a_dependency_diagram)
  * [The TopologyNode class](#the_topologynode_class)
  * [The Streams class](#the_streams_class)
* [The relationship between TopologyNode and Streams](#the_relationship_between_topologynode_and_streams)
* [How pydbsp does all the magic](#how_pydbsp_does_all_the_magic)


<a id="a_dependency_diagram"></a>
## A dependency diagram

Here is a dependency diagram:

```mermaid
---
title: Kafi Streams dependency diagram
---
classDiagram
    Streams <|-- TopologyNode
    Streams <|-- Kafi

    TopologyNode <|-- pydbsp
    TopologyNode <|-- msgpack
    TopologyNode <|-- cloudpickle
```

The two central classes in Kafi Streams are `TopologyNode` and `Streams`.


<a id="the_topologynode_class"></a>
### The TopologyNode class

The `TopologyNode` class is the fluent API on top of [*pydbsp*](https://github.com/brurucy/pydbsp) by Bruno Rucy, and is heavily inspired by the Kafka Streams DSL.

The class is completely abstracted away from Kafka. It does not know anything about Kafka. It just receives inputs, processes them relationally using pydbsp, and returns the outputs. This is why it can serve also as a test harness similar to the "TopologyTestDriver" in Kafka Streams.

These are the dependencies of the TopologyNode class, from bottom to top.

#### pydbsp

The by far most important building block is pydbsp by Bruno Rucy. It is the heart of Kafi Streams. It is the actual stream processing engine.

#### msgpack

In pydbsp, the fundamental data type is the *ZSet*. ZSets are implemented as dictionaries in pydbsp, where the keys are rows and the values are weights (=integers), e.g.:
```python
{"row_1": 1, "row_2": 0, "row_3": -1}
```

As Kafi Streams is typically used on top of Kafka where the payloads are encoded in JSON (and Kafi converts the JSONs into Python dictionaries automatically), I needed a fast way to serialize these dictionaries into a hashable form and deserialize them back to dictionaries.

This is the task of msgpack.

#### cloudpickle

cloudpickle is used for serializing/deserializing the global state of the topology (technically, the state of the pydbsp `evaluator`) since the built-in Python pickler is unable to serialize it.


<a id="the_streams_class"></a>
### The Streams class

The `Streams` subclass of `TopologyNode` adds support for Kafka.

#### Kafi

Kafi (the "old" part) provides all the Kafka support for Kafi Streams. It continuously consumes source topics, pushes the data to pydbsp, gets the outputs and produces them to sink topics.

Kafi also provides chunking/dechunking support which is required for checkpointing - where the checkpoints can go either to real Kafka or, through Kafi's "Kafka emulation", also to disk, S3 or Azure Blob Storage.


<a id="the_relationship_between_topologynode_and_streams"></a>
## The relationship between TopologyNode and Streams

What is the relationship between the two main classes of Kafi Streams, `TopologyNode` and `Streams`?

The [Quickstart](quickstart.ipynb) was based on the `Streams` subclass of the `TopologyNode` class because the aim was to give you the full picture from the start.

In the examples in the following chapters, however, for simplicity and brevity, we will often just use the `TopologyNode` class that has no connection with Kafka whatsoever.

`Streams`, as already alluded to above, is just about adding support for Kafka to `TopologyNode`. The actual stream processing in Kafi Streams is completely independent of Kafka - it could, in principle, be fed by any source and emit the output to any sink.

Here is a practical example - the code from the example in [Quickstart](quickstart.ipynb), but based on `TopologyNode` instead of `Streams`:

In [ ]:
# 1. Boilerplate

!pip install -r requirements.txt

import sys
sys.path.insert(1, "../..")

from kafi.streams.topologynode import TopologyNode as Tn

from generators import ClickGenerator, CustomerGenerator
click_generator = ClickGenerator()
customer_generator = CustomerGenerator()

import logging
logging.basicConfig(level=logging.INFO)

# 2. Specify the Topology

click_source_str = "clicks"
customer_source_str = "customers"
sink_str = "joined"

## a) Clicks

click_tn = (
    Tn.source(click_source_str)
    .map(lambda r: {"customer_id": r["value"]["customer_id"], "view_time": r["value"]["view_time"]})
    .filter(lambda r: r["view_time"] > 20)
    .distinct()
)

## b) Customers

customer_tn = (
    Tn.source(customer_source_str)
    .map(lambda r: {"id": r["value"]["id"], "name": r["value"]["name"]})
    .distinct()
)

## c) Join and Sink

sink_tn = (
    click_tn
    .join_equi(
        customer_tn,
        lambda l_r: l_r["customer_id"],
        lambda r_r: r_r["id"],
        lambda l_r, r_r: {"value": {
            "customer_id": l_r["customer_id"],
            "view_time": l_r["view_time"],
            "name": r_r["name"]}})
    .sink(sink_str)
)

# 3. Build the Topology

built_tn = Tn.build(sink_tn)


As you can see, the topology is defined identically. The only differences are:
* The connection to Kafka is left out, also in the sources and the sink specifications.
* The sources are specified using `Tn.source()` instead of `Streams.source()`.
* The build step is done using `Tn.build()` instead of `Streams.build()`.

Now how can supply data to the sources without Kafka? And how can we get the outputs of the processing?

Here is how we can use Kafi Streams without Kafka by using the `built_tn` object directly:

In [ ]:
sink_m_list = []
for i in range(100):
    # 1. Generate new data.
    click_m_list = click_generator.generate(100)
    customer_m_list = customer_generator.generate(100)

    # 2. Push the new data to the topology + incrementally process the new data + get the resulting changes.
    sink_str_m_dict = built_tn.process({click_source_str: click_m_list, customer_source_str: customer_m_list})

    # 3. Add the changes to the output list.
    sink_m_list += sink_str_m_dict.get(sink_str, [])

print(len(sink_m_list))
print(sink_m_list[-10:])


In this loop, we do the following:
1. We generate new data (10.000 clicks + 10.000 customers)
2. We push the new data to the topology using the `process()` method of the `TopologyNode` class. `process()` then processes the new data and returns the resulting changes.
3. We Add the changes to the output list.

That is exactly what the `Streams` subclass does, just with Kafka for sources and sinks.


<a id="how_pydbsp_does_all_the_magic"></a>
## How pydbsp does all the magic




* example operator
* evaluator
  * 2-dimensions
* input/output
